# Day 11: Your First Neural Network — MLP on MNIST 🎉

> **Goal:** Build a multi-layer perceptron (MLP) from scratch, train it end-to-end on MNIST, evaluate accuracy, and visualise results.

---

## The Big Picture

Today we combine everything from Days 1–10:

1. **Load data** (Day 10) → MNIST via `DataLoader`
2. **Define model** (Day 8) → `nn.Module` with `nn.Linear` layers
3. **Choose loss & optimizer** (Day 9) → `CrossEntropyLoss` + `Adam`
4. **Train** (Day 8) → the canonical loop
5. **Evaluate** → accuracy on the test set

By the end, you'll have a model that recognizes handwritten digits with >97% accuracy.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

---
## 1. Prepare the Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

full_train = torchvision.datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_data  = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# 80/20 train/val split
train_size = int(0.8 * len(full_train))
val_size = len(full_train) - train_size
train_data, val_data = random_split(full_train, [train_size, val_size],
                                     generator=torch.Generator().manual_seed(42))

BATCH_SIZE = 128
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

---
## 2. Define the MLP Model

An MLP for images:
1. **Flatten** the 28×28 image into a vector of 784
2. Pass through **hidden layers** with ReLU activations
3. Output **10 logits** (one per digit)

```
Input (784) → Linear(784, 256) → ReLU → Linear(256, 128) → ReLU → Linear(128, 10)
```

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=784, hidden1=256, hidden2=128, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()  # (B, 1, 28, 28) → (B, 784)
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, num_classes),
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.net(x)


model = MLP().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

---
## 3. Loss Function & Optimizer

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

---
## 4. Training & Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # Forward
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

---
## 5. Train! 🚀

In [ ]:
NUM_EPOCHS = 10

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.2%} | "
          f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.2%}")

### Training Curves

Plotting loss and accuracy over epochs helps you diagnose training. **What to look for:**
- Both curves should improve and eventually plateau
- If train keeps improving but val doesn't → overfitting (we'll fix this on Day 12)

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], label="Train")
ax1.plot(history["val_loss"], label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history["train_acc"], label="Train")
ax2.plot(history["val_acc"], label="Val")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy Curves")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Test Set Evaluation

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
print(f"\n🎯 Test Accuracy: {test_acc:.2%}")
print(f"   Test Loss:     {test_loss:.4f}")

---
## 7. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Collect all predictions
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=list(range(10)))

fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix — MLP on MNIST")
plt.tight_layout()
plt.show()

---
## 8. Visualize Predictions

In [ ]:
# Show some predictions
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
model.eval()

for i, ax in enumerate(axes.flat):
    img, true_label = test_data[i]
    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device)).argmax(1).item()

    ax.imshow(img.squeeze(), cmap="gray")
    color = "green" if pred == true_label else "red"
    ax.set_title(f"P:{pred} T:{true_label}", color=color, fontsize=10)
    ax.axis("off")

plt.suptitle("Predictions (Green=Correct, Red=Wrong)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 🧪 Exercises

### Exercise 1: Deeper network
Add a third hidden layer (e.g., 784 → 512 → 256 → 128 → 10). Does it improve accuracy? How many more parameters does it add?

### Exercise 2: Different optimizers
Train the same model with SGD (lr=0.01, momentum=0.9). Compare training curves with Adam.

### Exercise 3: Find the misclassified
Find and display 16 images that the model gets *wrong*. Are there patterns in what it confuses?

### Exercise 4: Per-class accuracy
Compute accuracy for each digit separately. Which digit is hardest to classify?

In [ ]:
# Exercise 1: Your code here

In [ ]:
# Exercise 2: Your code here

In [ ]:
# Exercise 3: Your code here

In [ ]:
# Exercise 4: Your code here

---
## 📝 Key Takeaways

| Concept | Summary |
|---|---|
| **MLP** | Flatten image → stack of Linear + ReLU → output logits |
| **Flatten** | Images must be reshaped from (C,H,W) to a 1D vector for an MLP |
| **Training loop** | forward → loss → zero_grad → backward → step |
| **Train/Val/Test** | Train on train, tune on val, report on test |
| **Confusion matrix** | Shows exactly which classes get confused |
| **~97%+ accuracy** | Achievable with a simple MLP on MNIST! |

**Limitation of MLPs for images:** Flattening throws away spatial structure. A pixel's neighbors carry important information. CNNs (Day 16+) will fix this!

**Tomorrow:** We'll learn about overfitting and regularization — how to make sure your model generalizes.